# Staged Pipeline Constraint Solver

## The Problem: Combinatorial Explosion

The flat N-candidate solver (see `demo/v2_n_candidate_demo.ipynb`) evaluates the full Cartesian product of all product roles. For LED lighting with 3 roles, that's **bars × drivers × dimmers** combinations. At catalog scale (100 × 40 × 50), this means 200,000 evaluations — and most of that work is wasted on combinations that fail early rules.

The staged pipeline solver fixes this by **decomposing the problem into sequential stages**, where each stage prunes invalid partial combinations before the next stage sees them.

**Reference:** See [`documentation/docs/architecture/multi-family-architecture.md`](../documentation/docs/architecture/multi-family-architecture.md) for the full architectural comparison.

## How It Works

Instead of one flat loop over all triples, the solver runs a pipeline:

```
Stage 1: light_bar × driver
  → Evaluate electrical rules (LED001-003, 006-008)
  → Keep only valid bar-driver pairs
  → PRUNE: invalid pairs never reach Stage 2

Stage 2: valid_pairs × dimmer
  → Evaluate dimming rules (LED004, 005, 009)
  → Keep only valid triples
  → Final result
```

Each stage introduces **new product roles** and evaluates **stage-specific rules**. The key insight: if a bar-driver pair fails voltage match (LED001), that failure is independent of which dimmer you pair it with. There's no reason to re-evaluate it with all 50 dimmers.

## Complexity

**Time:** O(A × B × R₁) + O(valid_AB × C × R₂)

If Stage 1 has a **pruning rate** of P (fraction of pairs eliminated), the savings are:

```
Flat:    A × B × C × R     evaluations
Staged:  A × B × R₁  +  (1-P) × A × B × C × R₂   evaluations
```

When P = 80% (common for hardware catalogs where voltage/connector mismatches dominate):
- Flat:    100 × 40 × 50 × 9 = 1,800,000 rule evaluations
- Staged:  100 × 40 × 6  +  20% × 100 × 40 × 50 × 3 = 24,000 + 120,000 = 144,000
- **Savings: 92%**

In [ ]:
import sys, time, random
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

from engine_v2.core.solver_staged import StagedPipelineSolver, StagedFamilyConfig, Stage
from engine_v2.core.solver_n import NCandidateSolver, NFamilyConfig
from engine_v2.families.led_lighting.models import (
    LightBar, Driver, Dimmer, LightingRequirements,
    Voltage, DimmingProtocol, ConnectorType,
)
from engine_v2.families.led_lighting.rules import ALL_RULES, STAGE_1_RULES, STAGE_2_RULES
from engine_v2.families.led_lighting.test_data import (
    ALL_BARS, ALL_DRIVERS, ALL_DIMMERS,
    BAR_12V_5W, BAR_12V_10W, BAR_24V_15W, BAR_24V_8W_DIM, BAR_12V_LONG,
    DRV_12V_30W, DRV_12V_15W_NODIM, DRV_24V_60W, DRV_24V_20W,
    DIM_TRAILING_150W, DIM_0_10V_200W, DIM_TRAILING_SMALL, DIM_LEADING,
)

# Configure the staged family
staged_config = StagedFamilyConfig(
    name="led_lighting_staged",
    roles=[
        ("light_bar", LightBar),
        ("driver", Driver),
        ("dimmer", Dimmer),
    ],
    requirements_type=LightingRequirements,
    stages=[
        Stage(
            name="electrical_compatibility",
            new_roles=["light_bar", "driver"],
            rules=STAGE_1_RULES,
            early_termination=True,
        ),
        Stage(
            name="dimming_compatibility",
            new_roles=["dimmer"],
            rules=STAGE_2_RULES,
            early_termination=True,
        ),
    ],
    rank_key=lambda c: (0 if c.total_price_usd is not None else 1, c.total_price_usd or 0),
)

solver = StagedPipelineSolver(
    config=staged_config,
    product_lists={
        "light_bar": ALL_BARS,
        "driver": ALL_DRIVERS,
        "dimmer": ALL_DIMMERS,
    },
)

print("STAGED PIPELINE CONFIGURATION")
print("=" * 70)
for i, stage in enumerate(staged_config.stages, 1):
    print(f"\nStage {i}: {stage.name}")
    print(f"  Introduces roles: {stage.new_roles}")
    print(f"  Rules ({len(stage.rules)}):")
    # Get rule IDs by calling with dummy data
    for rule in stage.rules:
        dummy = {"light_bar": ALL_BARS[0], "driver": ALL_DRIVERS[0], "dimmer": ALL_DIMMERS[0]}
        r = rule(dummy, LightingRequirements(cabinet_length_mm=600), {})
        print(f"    {r.rule_id} — {r.rule_name}")

print(f"\nTotal: {len(STAGE_1_RULES) + len(STAGE_2_RULES)} rules across {len(staged_config.stages)} stages")
print(f"Catalog: {len(ALL_BARS)} bars x {len(ALL_DRIVERS)} drivers x {len(ALL_DIMMERS)} dimmers")

## 1. Stage-by-Stage Visualization

Let's trace the pipeline to see what happens at each stage. The key metric is the **pruning rate** — what percentage of combinations are eliminated before reaching the next stage.

In [ ]:
# Manually trace Stage 1 to show the pruning
req = LightingRequirements(cabinet_length_mm=600, dimming_required=True)

print("STAGE 1: Electrical Compatibility (light_bar x driver)")
print("=" * 80)
print(f"{'Light Bar':<22} {'Driver':<22} {'Result':<8} {'Failed Rule':<30}")
print("-" * 80)

stage1_valid = []
stage1_total = 0

for bar in ALL_BARS:
    for drv in ALL_DRIVERS:
        stage1_total += 1
        all_pass = True
        failed = ""
        for rule in STAGE_1_RULES:
            candidates = {"light_bar": bar, "driver": drv, "dimmer": ALL_DIMMERS[0]}
            r = rule(candidates, req, {})
            if not r.passed and r.category.value == "hard_constraint":
                all_pass = False
                failed = f"{r.rule_id} {r.rule_name}"
                break
        
        status = "PASS" if all_pass else "FAIL"
        print(f"{bar.sku:<22} {drv.sku:<22} {status:<8} {failed:<30}")
        if all_pass:
            stage1_valid.append((bar, drv))

pruning_rate = (1 - len(stage1_valid) / stage1_total) * 100
print(f"\nStage 1 results: {len(stage1_valid)}/{stage1_total} pairs passed ({pruning_rate:.0f}% pruned)")
print(f"\n--- PRUNING IMPACT ---")
print(f"Without pruning: {stage1_total} pairs x {len(ALL_DIMMERS)} dimmers = {stage1_total * len(ALL_DIMMERS)} triples enter Stage 2")
print(f"With pruning:    {len(stage1_valid)} pairs x {len(ALL_DIMMERS)} dimmers = {len(stage1_valid) * len(ALL_DIMMERS)} triples enter Stage 2")
print(f"Saved: {(stage1_total - len(stage1_valid)) * len(ALL_DIMMERS)} unnecessary evaluations")

In [ ]:
# Stage 2: only valid pairs from Stage 1 get crossed with dimmers
print("STAGE 2: Dimming Compatibility (valid_pair x dimmer)")
print("=" * 90)
print(f"{'Light Bar':<22} {'Driver':<22} {'Dimmer':<22} {'Result':<8} {'Failed Rule'}")
print("-" * 90)

stage2_valid = 0
stage2_total = 0

for bar, drv in stage1_valid:
    for dim in ALL_DIMMERS:
        stage2_total += 1
        all_pass = True
        failed = ""
        for rule in STAGE_2_RULES:
            candidates = {"light_bar": bar, "driver": drv, "dimmer": dim}
            r = rule(candidates, req, {})
            if not r.passed and r.category.value == "hard_constraint":
                all_pass = False
                failed = f"{r.rule_id} {r.rule_name}"
                break
        
        status = "PASS" if all_pass else "FAIL"
        print(f"{bar.sku:<22} {drv.sku:<22} {dim.sku:<22} {status:<8} {failed}")
        if all_pass:
            stage2_valid += 1

print(f"\nStage 2 results: {stage2_valid}/{stage2_total} triples passed")
print(f"\nPIPELINE SUMMARY:")
print(f"  Stage 1: {stage1_total} pairs evaluated → {len(stage1_valid)} valid")
print(f"  Stage 2: {stage2_total} triples evaluated → {stage2_valid} valid")
print(f"  Total evaluations: {stage1_total + stage2_total}")
print(f"  Flat approach would evaluate: {stage1_total * len(ALL_DIMMERS)} triples")
print(f"  Savings: {stage1_total * len(ALL_DIMMERS) - stage1_total - stage2_total} fewer evaluations")

## 2. Solving Scenarios

The staged solver produces **identical results** to the flat N-candidate solver — same valid configurations, same ranking, same constraint traces. The traces include rule results from **all stages**, so the conversational layer sees one unified trace regardless of which solver was used.

This equivalence is verified by 5 cross-solver consistency tests in `engine_v2/tests/test_staged.py::TestCrossSolverConsistency`.

In [ ]:
# Scenario: 600mm cabinet, dimming required
req = LightingRequirements(
    cabinet_length_mm=600,
    dimming_required=True,
    min_lumen_output=300,
)

result = solver.solve_with_explanation(req)

print(f"Status: {result['status']}")
print(f"{result['message']}\n")

if result["status"] == "solved":
    rec = result["recommended"]
    print("RECOMMENDED CONFIGURATION:")
    for role, prod in rec["candidates"].items():
        print(f"  {role:>12}: {prod['sku']}")
    price = f"${rec['total_price_usd']:.2f}" if rec['total_price_usd'] else "N/A"
    print(f"  {'total price':>12}: {price}")
    
    print(f"\nConstraint trace ({len(rec['constraint_trace'])} rules from both stages):")
    for rule in rec["constraint_trace"]:
        icon = "PASS" if rule["passed"] else "FAIL"
        print(f"  [{icon}] {rule['rule']} {rule['name']}: {rule['detail']}")
    
    print(f"\n+ {len(result['alternatives'])} alternative(s)")
    
    # Show all valid configs
    valid = solver.solve(req)
    print(f"\nAll {len(valid)} valid configurations:")
    print(f"{'#':>3}  {'Light Bar':<22} {'Driver':<22} {'Dimmer':<22} {'Price':>8}")
    print("-" * 80)
    for i, config in enumerate(valid, 1):
        bar = config.candidates["light_bar"]
        drv = config.candidates["driver"]
        dim = config.candidates["dimmer"]
        p = f"${config.total_price_usd:.2f}" if config.total_price_usd else "N/A"
        print(f"{i:>3}  {bar.sku:<22} {drv.sku:<22} {dim.sku:<22} {p:>8}")

## 3. Failure Analysis

The staged solver's `_best_failing()` method temporarily disables pruning and evaluates the full Cartesian product to find the closest match. This is necessary because the closest-to-valid combination might have been pruned at Stage 1.

This means failure analysis is O(A × B × C) regardless of solver — the staged approach only saves work when valid solutions exist.

In [ ]:
# Impossible scenario — compare with v2_n_candidate_demo.ipynb
fail_req = LightingRequirements(
    cabinet_length_mm=1200,
    dimming_required=True,
    min_lumen_output=1500,
    voltage_preference=Voltage.DC_12V,
)

fail_result = solver.solve_with_explanation(fail_req)

print(f"Status: {fail_result['status']}")
print(f"{fail_result['message']}\n")

if fail_result.get("closest_match"):
    cm = fail_result["closest_match"]
    print(f"Closest match:")
    for role, prod in cm["candidates"].items():
        print(f"  {role}: {prod['sku']}")

print(f"\nFailed constraints:")
for fr in fail_result["failed_rules"]:
    print(f"  [{fr['rule']}] {fr['name']}: {fr['detail']}")
    if fr.get("remediation"):
        print(f"           Fix: {fr['remediation']}")

print(f"\nNote: _best_failing() evaluated ALL {len(ALL_BARS) * len(ALL_DRIVERS) * len(ALL_DIMMERS)} combinations")
print(f"to find this closest match — no pruning during failure analysis.")

## 4. Head-to-Head Benchmark: Staged vs Flat

This is the key comparison. Both solvers produce identical results, but the staged solver does less work by pruning invalid partial combinations.

We benchmark both at increasing catalog sizes using synthetic product data. The pruning advantage grows with catalog size — at small scales the overhead of stage management can actually make the staged solver slightly slower, but at scale the savings dominate.

In [ ]:
# Synthetic data generators (same as v2_n_candidate_demo.ipynb)
def generate_synthetic_bars(n):
    bars = []
    for i in range(n):
        bars.append(LightBar(
            sku=f"BAR-{i:04d}", brand=random.choice(["Loox", "Hafele", "Domus"]),
            price_usd=round(random.uniform(10, 80), 2),
            wattage=random.choice([5, 8, 10, 15, 20]),
            voltage=random.choice([Voltage.DC_12V, Voltage.DC_24V]),
            length_mm=random.choice([300, 450, 600, 900, 1200]),
            lumen_output=random.choice([300, 500, 800, 1200, 1600]),
            dimmable=random.choice([True, True, False]),
            connector=random.choice([ConnectorType.BARREL_JACK, ConnectorType.TERMINAL_BLOCK]),
        ))
    return bars

def generate_synthetic_drivers(n):
    drivers = []
    for i in range(n):
        v = random.choice([Voltage.DC_12V, Voltage.DC_24V])
        conn = ConnectorType.BARREL_JACK if v == Voltage.DC_12V else ConnectorType.TERMINAL_BLOCK
        dimmable = random.choice([True, True, False])
        proto = random.choice([DimmingProtocol.TRAILING_EDGE, DimmingProtocol.ZERO_TO_10V]) if dimmable else DimmingProtocol.NONE
        drivers.append(Driver(
            sku=f"DRV-{i:04d}", brand=random.choice(["Loox", "Hafele", "Mean Well"]),
            price_usd=round(random.uniform(10, 60), 2),
            output_voltage=v, max_wattage=random.choice([15, 30, 60, 100]),
            output_channels=random.choice([2, 4, 6]), dimmable=dimmable,
            dimming_protocol=proto, connector=conn,
        ))
    return drivers

def generate_synthetic_dimmers(n):
    dimmers = []
    for i in range(n):
        proto = random.choice([DimmingProtocol.TRAILING_EDGE, DimmingProtocol.ZERO_TO_10V, DimmingProtocol.LEADING_EDGE])
        dimmers.append(Dimmer(
            sku=f"DIM-{i:04d}", brand=random.choice(["Loox", "Hafele", "Lutron"]),
            price_usd=round(random.uniform(15, 80), 2),
            dimming_protocol=proto, max_wattage=random.choice([50, 100, 150, 200]),
            voltage_compatible=random.choice([[Voltage.DC_12V], [Voltage.DC_24V], [Voltage.DC_12V, Voltage.DC_24V]]),
            min_load_wattage=random.choice([0, 2, 5, 10]),
        ))
    return dimmers

# N-candidate config (for comparison)
flat_config = NFamilyConfig(
    name="led_flat",
    roles=[("light_bar", LightBar), ("driver", Driver), ("dimmer", Dimmer)],
    requirements_type=LightingRequirements,
    rules=ALL_RULES,
    early_termination=True,
)

# Benchmark at different scales
random.seed(42)
req = LightingRequirements(cabinet_length_mm=600, dimming_required=True)

print("HEAD-TO-HEAD BENCHMARK: Staged Pipeline vs Flat N-Candidate")
print("=" * 100)
print(f"{'Scale':<25} {'Combos':>10} {'Flat (ms)':>10} {'Staged (ms)':>12} {'Speedup':>8} {'Both Valid':>11}")
print("-" * 100)

benchmark_sizes = [
    ("5 x 4 x 4",           5,    4,    4),
    ("20 x 10 x 10",       20,   10,   10),
    ("50 x 20 x 30",       50,   20,   30),
    ("100 x 40 x 50",     100,   40,   50),
    ("200 x 80 x 100",    200,   80,  100),
]

for label, nb, nd, ndm in benchmark_sizes:
    bars = generate_synthetic_bars(nb) if nb > 5 else ALL_BARS
    drivers = generate_synthetic_drivers(nd) if nd > 4 else ALL_DRIVERS
    dimmers = generate_synthetic_dimmers(ndm) if ndm > 4 else ALL_DIMMERS
    products = {"light_bar": bars, "driver": drivers, "dimmer": dimmers}
    
    # Flat solver
    flat_solver = NCandidateSolver(config=flat_config, product_lists=dict(products))
    start = time.perf_counter()
    flat_results = flat_solver.solve(req)
    flat_ms = (time.perf_counter() - start) * 1000
    
    # Staged solver
    staged_solver = StagedPipelineSolver(config=staged_config, product_lists=dict(products))
    start = time.perf_counter()
    staged_results = staged_solver.solve(req)
    staged_ms = (time.perf_counter() - start) * 1000
    
    combos = nb * nd * ndm
    speedup = flat_ms / staged_ms if staged_ms > 0 else float('inf')
    same_count = len(flat_results) == len(staged_results)
    
    print(f"{label:<25} {combos:>10,} {flat_ms:>9.1f}ms {staged_ms:>11.1f}ms {speedup:>7.1f}x {'YES' if same_count else 'NO':>11}")

print(f"\n'Both Valid' confirms both solvers return the same number of valid configurations.")

## 5. Pruning Rate Analysis

The staged solver's advantage depends entirely on how many combinations Stage 1 eliminates. If Stage 1 prunes nothing (all bar-driver pairs are valid), the staged solver does the same work as the flat solver plus stage management overhead.

The pruning rate depends on the **constraint density** of the catalog — how many products are incompatible with each other. Hardware catalogs tend to have high constraint density because:
- Voltage splits the catalog in half (12V vs 24V)
- Connector types create another split
- Non-dimmable drivers eliminate dimming configurations

Let's measure the actual pruning rate at different catalog sizes.

In [ ]:
# Measure Stage 1 pruning rate at different scales
random.seed(42)
req = LightingRequirements(cabinet_length_mm=600, dimming_required=True)

print("PRUNING RATE ANALYSIS — Stage 1 (bar x driver)")
print("=" * 95)
print(f"{'Scale':<25} {'Pairs':>8} {'Valid':>6} {'Pruned':>7} {'Rate':>6}  {'Flat triples':>13} {'Staged eval':>12} {'Savings':>8}")
print("-" * 95)

for label, nb, nd, ndm in benchmark_sizes:
    bars = generate_synthetic_bars(nb) if nb > 5 else ALL_BARS
    drivers = generate_synthetic_drivers(nd) if nd > 4 else ALL_DRIVERS
    dimmers = generate_synthetic_dimmers(ndm) if ndm > 4 else ALL_DIMMERS
    
    # Count valid bar-driver pairs
    valid_pairs = 0
    total_pairs = len(bars) * len(drivers)
    
    for bar in bars:
        for drv in drivers:
            pair_valid = True
            for rule in STAGE_1_RULES:
                candidates = {"light_bar": bar, "driver": drv, "dimmer": dimmers[0]}
                r = rule(candidates, req, {})
                if not r.passed and r.category.value == "hard_constraint":
                    pair_valid = False
                    break
            if pair_valid:
                valid_pairs += 1
    
    pruned = total_pairs - valid_pairs
    rate = pruned / total_pairs * 100 if total_pairs > 0 else 0
    flat_triples = total_pairs * len(dimmers)
    staged_evals = total_pairs + valid_pairs * len(dimmers)  # Stage 1 + Stage 2
    savings = (1 - staged_evals / flat_triples) * 100 if flat_triples > 0 else 0
    
    print(f"{label:<25} {total_pairs:>8,} {valid_pairs:>6,} {pruned:>7,} {rate:>5.0f}%  {flat_triples:>13,} {staged_evals:>12,} {savings:>7.0f}%")

print(f"\n'Staged eval' = Stage 1 pairs + (valid pairs x dimmers)")
print(f"'Savings' = reduction in total evaluations compared to flat approach")
print(f"\nThe higher the pruning rate, the more the staged solver saves.")
print(f"At 50%+ pruning (typical for multi-voltage catalogs), staged is 2-4x faster.")

## 6. Stage Ordering Matters

The staged pipeline's effectiveness depends on putting the **most restrictive constraints first**. If Stage 1 prunes 80% of pairs, Stage 2 sees 5x fewer combinations. If Stage 1 prunes nothing, the pipeline adds overhead for no benefit.

For LED lighting, the natural ordering is:

1. **Electrical compatibility first** (bar × driver) — voltage and connector mismatches eliminate ~50-80% of pairs
2. **Dimming compatibility second** (valid pair × dimmer) — protocol and wattage checks on the survivors

What if we reversed the order? Stage 1 would be driver × dimmer (dimming protocol), and Stage 2 would add the light bar. This would be less effective because:
- Dimming protocol only splits dimmers into 3 groups (trailing-edge, 0-10V, leading-edge)
- Voltage mismatch (the biggest pruner) wouldn't apply until Stage 2

### The cross-cutting constraint problem

Some rules need products from **different stages**. LED005 (dimmer wattage) needs both the light bar (from Stage 1) and the dimmer (from Stage 2). In the staged pipeline, this works because Stage 2 has access to all products accumulated from prior stages. But if a rule needed products that are introduced in the **same stage from different pairs**, it wouldn't fit.

This is the main design limitation of the staged approach — see [`documentation/docs/architecture/multi-family-architecture.md`](../documentation/docs/architecture/multi-family-architecture.md) for discussion.

## 7. When to Use This Approach

The staged pipeline is the right choice when:

- **Constraints are layered.** Some products constrain each other but not all products constrain all others. LED lighting is a good example: bar-dimmer has no direct constraints.
- **Catalogs are large.** At 200 × 80 × 100 = 1.6M combinations, pruning is essential.
- **Adding more product roles.** A 4th role adds one more stage, not another nested loop dimension. The pipeline grows linearly, not exponentially.
- **Latency matters.** API endpoints serving real-time recommendations benefit from 2-10x fewer evaluations.

It is **not** the right choice when:

- **All products constrain each other.** If every pair of roles has rules between them, Stage 1 can't prune anything because it doesn't know about the products in Stage 2. The overhead of stage management adds cost for no benefit.
- **Catalogs are small.** Under ~1,000 combinations, the flat solver is faster due to less overhead.
- **You need complete evaluation data.** The staged solver only fully evaluates combinations that survive all stages. For analytics or compatibility matrices, the flat solver gives you the full picture.
- **Failure analysis is the common case.** `_best_failing()` evaluates everything anyway — no pruning benefit.

## Summary

| | Flat N-Candidate | Staged Pipeline |
|---|---|---|
| **Complexity** | O(A × B × C × R) | O(A × B × R₁) + O(valid × C × R₂) |
| **Pruning** | None | Between stages |
| **Best for** | Small catalogs, analytics | Large catalogs, APIs |
| **Config effort** | One rule list | Stage decomposition + rule assignment |
| **Failure analysis** | Same cost | Same cost (no pruning) |
| **Results** | Identical | Identical |

**Source code:** `engine_v2/core/solver_staged.py` (solver), `engine_v2/families/led_lighting/` (shared models + rules), `engine_v2/tests/test_staged.py` (25 tests + 5 cross-solver consistency tests)

**Design document:** [`documentation/docs/architecture/multi-family-architecture.md`](../documentation/docs/architecture/multi-family-architecture.md)

**Companion notebook:** `demo/v2_n_candidate_demo.ipynb` (flat approach with the same scenarios)